In [1]:
import keras
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
from keras import layers
import numpy as np
from keras.saving import register_keras_serializable
from sklearn.metrics import confusion_matrix
import pandas as pd
import os, random



In [11]:
base_dir=r"D:\training\skenario\kedua"
img_size=(244, 244)
batch_size= 32


In [12]:
train_ds= tf.keras.utils.image_dataset_from_directory(
    base_dir +"/train",
    image_size= img_size,
    batch_size= batch_size,
    label_mode= "categorical",
    shuffle= True
)

Found 3200 files belonging to 4 classes.


In [13]:
test_ds= tf.keras.utils.image_dataset_from_directory(
    base_dir+"/test",
    image_size= img_size,
    batch_size= batch_size,
    label_mode= "categorical",
    shuffle= False
)

Found 500 files belonging to 4 classes.


In [14]:
val_ds= tf.keras.utils.image_dataset_from_directory(
    base_dir+"/val",
    image_size= img_size,
    batch_size= batch_size,
    label_mode= "categorical",
    shuffle= False
)

Found 600 files belonging to 4 classes.


In [52]:
@register_keras_serializable()
class MyModel(keras.Model):
    def __init__(self):
        super().__init__()
        self.conv2d1= keras.layers.Conv2D(
            32, kernel_size=(3, 3), activation='relu', padding='same'
        )
        self.pool1= keras.layers.MaxPool2D((2, 1))
        self.batch1= keras.layers.BatchNormalization()


        self.conv2d2= keras.layers.Conv2D(
            64, kernel_size=(3, 3), padding="same", activation='relu'
        )

        self.pool2= keras.layers.MaxPool2D((2, 1))


        self.batch2= keras.layers.BatchNormalization()
        self.drop2= keras.layers.Dropout(0.1)

        self.reshape= keras.layers.Reshape(
            (244, 3904)
        )
        self.lstm1= keras.layers.LSTM(
            units=128, return_sequences=True
        )
        self.lstm2= keras.layers.LSTM(
            units=64
        )
        self.dense1= keras.layers.Dense(
            64, activation='relu'
        )

        self.dense2= keras.layers.Dense(
            4, activation='softmax'
        )

    def call(self, x, training=False):
        x= self.conv2d1(x)
        x= self.pool1(x)
        
        x= self.batch1(x, training=training)
        
        x= self.conv2d2(x)
        x= self.pool2(x)
    
        x= self.batch2(x, training=training)

        x= self.drop2(x, training=training)

        x= tf.transpose(x, [0, 2, 1, 3])
        x= self.reshape(x)

        
        x= self.lstm1(x)
        x= self.lstm2(x)
        x= self.dense1(x)
        return self.dense2(x)



In [53]:
model= MyModel()

opt= keras.optimizers.Adam(learning_rate=1e-3)

model.compile(
    optimizer= opt,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

class_weight= {
    0: 1.1,
    1: 1.2,
    2: 1.1,
    3: 1.2
}



In [54]:
early_stop= EarlyStopping(
    monitor="val_loss",
    patience= 15,
    restore_best_weights=True   
)

In [55]:
model.fit(
    train_ds,
    validation_data =val_ds,
    epochs=30,
    class_weight= class_weight,
    callbacks= early_stop
)  

Epoch 1/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 604s 6s/step - accuracy: 0.7600 - loss: 0.7189 - val_accuracy: 0.1750 - val_loss: 1.3860
Epoch 2/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 625s 6s/step - accuracy: 0.9384 - loss: 0.1959 - val_accuracy: 0.1600 - val_loss: 1.6312
Epoch 3/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 707s 7s/step - accuracy: 0.9703 - loss: 0.1040 - val_accuracy: 0.3650 - val_loss: 1.6794
Epoch 4/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 606s 6s/step - accuracy: 0.9862 - loss: 0.0494 - val_accuracy: 0.4067 - val_loss: 1.4046
Epoch 5/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 696s 7s/step - accuracy: 0.9894 - loss: 0.0365 - val_accuracy: 0.7117 - val_loss: 0.7706
Epoch 6/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 549s 5s/step - accuracy: 0.9831 - loss: 0.0612 - val_accuracy: 0.5333 - val_loss: 2.5981
Epoch 7/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 502s 5s/step - accuracy: 0.9934 - loss: 0.0212 - val_accuracy: 0.8967 - val_loss: 0.4274
Epoch 8/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 499s 5s/step - accuracy: 0.9947 - loss: 0.0234 - val_accu

KeyboardInterrupt: 

In [19]:
y_true_oh= np.concatenate([y.numpy() for x, y in test_ds], axis=0 )
y_true= np.argmax(y_true_oh, axis=1)
y_pred= np.argmax(model.predict(test_ds), axis=1)

label= ['Conversion', 'Real', 'Record', 'Syntesys']
cm= confusion_matrix(y_true, y_pred)
cm_df= pd.DataFrame(cm, index=label, columns=label)

print(cm_df)

Before LSTM (32, 244, 3904)
15/16 ━━━━━━━━━━━━━━━━━━━━ 0s 692ms/stepBefore LSTM (None, 244, 3904)
16/16 ━━━━━━━━━━━━━━━━━━━━ 13s 709ms/step
            Conversion  Real  Record  Syntesys
Conversion         198     0       0         2
Real                 0   100       0         0
Record               0     0      99         1
Syntesys             1     0       0        99


In [16]:
model.save("model_crnn.keras")